In [ ]:
from functools import reduce

import colorcet as cc
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from utils import RESULT_DIR

In [ ]:
results_abc = RESULT_DIR / "ABC_atlas"

In [ ]:
method = "scvi"

In [ ]:
ref = "1.0"
low = "0.5"
high = "1.5"

In [ ]:
labels = []

for file in results_abc.glob(f"{method}_r*.parquet"):
    resolution = file.stem.split("_")[1][1:]
    labels.append(
        pd.read_parquet(file, columns=["cell_label", "spatialleiden"]).rename(
            columns={"spatialleiden": resolution}
        )
    )


labels = reduce(
    lambda df1, df2: pd.merge(df1, df2, left_index=True, right_index=True), labels
)

In [ ]:
n_ref = labels[ref].nunique()
n_low = labels[low].nunique()
n_high = labels[high].nunique()

col_ref = cc.glasbey[:n_ref]
col_low = cc.glasbey[n_ref : n_ref + n_low]
col_high = cc.glasbey[n_ref : n_ref + n_low + n_high]

In [ ]:
# threshold on how many cells need to change from cluster A -> B to show the connection
min_cells = 1_000  # this is only ~ 0.04 % of overall cells

counts = labels.groupby([low, ref], observed=True).size().to_frame("n").reset_index()
counts = counts.loc[lambda df: df["n"] > min_cells]

sources_lr = counts[low].tolist()
targets_lr = (counts[ref] + n_low).tolist()
values_lr = counts["n"].tolist()
color_lr = list(np.array(col_low)[sources_lr])


counts = labels.groupby([ref, high], observed=True).size().to_frame("n").reset_index()
counts = counts.loc[lambda df: df["n"] > min_cells]

sources_rh = (counts[ref] + n_low).tolist()
targets_rh = (counts[high] + n_low + n_ref).tolist()
values_rh = counts["n"].tolist()
color_rh = list(np.array(col_ref)[counts[ref].to_numpy()])

In [ ]:
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(pad=5, thickness=10, color=col_low + col_ref + col_high),
            link=dict(
                source=sources_lr + sources_rh,
                target=targets_lr + targets_rh,
                value=values_lr + values_rh,
                color=color_lr + color_rh,
            ),
        )
    ]
)

fig.update_layout(font_size=10)
fig

In [ ]:
fig.write_image(results_abc / "sankey.png")
fig.write_image(results_abc / "sankey.pdf")